# NQ100 売買ルールバックテスト

Google Drive上のCSVを読み込み、東京時間の市場状態AIと売買シミュレーションを検証するためのColabノートブックです。

In [57]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/NQ100')
REPO_URL = 'https://github.com/hon-daisuki/NQ100.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)

repo_path = str(REPO_DIR)
if repo_path in sys.path:
    sys.path.remove(repo_path)
sys.path.insert(0, repo_path)

for module_name in list(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
import pandas as pd
import numpy as np

from src.labels import add_future_return_label, add_direction_label
from src.backtest import build_long_short_returns, equity_curve, summarize_returns, yearly_returns
from src.metrics import describe_time_range, missing_summary

## データ読み込み

In [59]:
DATA_ROOT = Path('/content/drive')
CSV_CANDIDATES = [
    'USTEC_features_all.csv',
    'USTEC_1hour_features.csv',
]
PREFERRED_DIRS = [
    Path('/content/drive/MyDrive/CFD機械学習/backtest_ready'),
    Path('/content/drive/MyDrive/CFD機械学習'),
    Path('/content/drive/MyDrive'),
    DATA_ROOT,
]

def find_features_csv():
    checked = []
    for folder in PREFERRED_DIRS:
        for name in CSV_CANDIDATES:
            candidate = folder / name
            checked.append(str(candidate))
            if candidate.exists():
                return candidate

    matches = []
    for name in CSV_CANDIDATES:
        matches.extend(DATA_ROOT.rglob(name))
    if matches:
        return sorted(matches, key=lambda x: len(str(x)))[0]

    raise FileNotFoundError(
        '特徴量CSVが見つかりません。Driveに対象CSVがあるか、共有フォルダをMyDriveにショートカット追加してください。\n'
        + '探した候補:\n' + '\n'.join(checked)
    )

csv_path = find_features_csv()
print(f'Using CSV: {csv_path}')

df = pd.read_csv(csv_path)
df['日時'] = pd.to_datetime(df['日時'])
df = df.sort_values('日時').reset_index(drop=True)

describe_time_range(df)

Using CSV: /content/drive/MyDrive/CFD機械学習/NQ100/USTEC_1hour_features.csv


{'rows': 85831,
 'start': Timestamp('2011-10-01 00:00:00'),
 'end': Timestamp('2026-05-23 05:00:00'),
 'columns': ['日時',
  'open',
  'high',
  'low',
  'close',
  'return_1h',
  'range',
  'body',
  'upper_wick',
  'lower_wick',
  'hour',
  'dayofweek',
  'month',
  'sma_5',
  'sma_20',
  'ema_12',
  'ema_26',
  'macd',
  'macd_signal',
  'macd_hist',
  'bb_mid',
  'bb_std',
  'bb_upper',
  'bb_lower',
  'bb_width',
  'rsi_14',
  'atr_14']}

In [60]:
missing_summary(df)

,missing,missing_rate
bb_mid,19,0.000221
sma_20,19,0.000221
bb_lower,19,0.000221
bb_width,19,0.000221
bb_std,19,0.000221
bb_upper,19,0.000221
rsi_14,14,0.000163
atr_14,13,0.000151
sma_5,4,0.000047
return_1h,1,0.000012


## 東京時間フィルタとラベル作成

In [61]:
df['hour'] = df['日時'].dt.hour
df_tokyo = df[(df['hour'] >= 8) & (df['hour'] <= 12)].copy()

df_tokyo = add_future_return_label(df_tokyo, horizon=4, threshold=0.005)
df_tokyo = add_direction_label(df_tokyo, return_col='future_return_4h', threshold=0.005)

df_tokyo[['日時', 'close', 'future_return_4h', 'is_trend', 'signal']].tail()

,日時,close,future_return_4h,is_trend,signal
85809,2026-05-22 08:00:00,29549.9,0.000741,0,0
85810,2026-05-22 09:00:00,29564.5,NaN,0,0
85811,2026-05-22 10:00:00,29528.8,NaN,0,0
85812,2026-05-22 11:00:00,29548.1,NaN,0,0
85813,2026-05-22 12:00:00,29571.8,NaN,0,0


## 学習データ

In [62]:
features_trend = [
    'return_1h', 'range', 'body', 'upper_wick', 'lower_wick',
    'hour', 'dayofweek', 'month', 'sma_5', 'sma_20',
    'macd', 'macd_signal', 'macd_hist', 'bb_width', 'rsi_14', 'atr_14'
]

df_model = df_tokyo.dropna(subset=features_trend + ['is_trend', 'future_return_4h']).copy()
split_index = int(len(df_model) * 0.8)

X_train = df_model[features_trend].iloc[:split_index]
X_test = df_model[features_trend].iloc[split_index:]
y_train = df_model['is_trend'].iloc[:split_index]
y_test = df_model['is_trend'].iloc[split_index:]

len(X_train), len(X_test), y_train.mean(), y_test.mean()

(15000, 3751, np.float64(0.4706), np.float64(0.49853372434017595))

## LightGBM学習

In [63]:
!pip -q install lightgbm

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_trend = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_trend.fit(X_train, y_train)
proba = model_trend.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('Accuracy =', accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

[LightGBM] [Info] Number of positive: 7059, number of negative: 7941
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000790 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3339
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Accuracy = 0.6475606504932018
[[ 996  885]
 [ 437 1433]]
              precision    recall  f1-score   support

           0       0.70      0.53      0.60      1881
           1       0.62      0.77      0.68      1870

    accuracy                           0.65      3751
   macro avg       0.66      0.65      0.64      3751
weighted avg       0.66      0.65      0.64      3751



## Direction Model

Predict whether the future 4-hour move is down, flat, or up.

In [64]:
direction_threshold = 0.005

df_model['direction_signal'] = 0
df_model.loc[df_model['future_return_4h'] > direction_threshold, 'direction_signal'] = 1
df_model.loc[df_model['future_return_4h'] < -direction_threshold, 'direction_signal'] = -1

# Convert -1/0/1 signals to 0/1/2 classes for LightGBM multiclass training.
signal_to_class = {-1: 0, 0: 1, 1: 2}
class_to_signal = {0: -1, 1: 0, 2: 1}

y_dir = df_model['direction_signal'].map(signal_to_class)
y_dir_train = y_dir.iloc[:split_index]
y_dir_test = y_dir.iloc[split_index:]

df_model['direction_signal'].value_counts(normalize=True).sort_index()


,proportion
direction_signal,
-1,0.204096
0,0.523812
1,0.272092


In [65]:
model_dir = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_dir.fit(X_train, y_dir_train)
dir_proba = model_dir.predict_proba(X_test)
dir_class_pred = dir_proba.argmax(axis=1)
dir_signal_pred = pd.Series(dir_class_pred, index=X_test.index).map(class_to_signal)

print('Direction Accuracy =', accuracy_score(y_dir_test, dir_class_pred))
print(confusion_matrix(y_dir_test, dir_class_pred))
print(classification_report(
    y_dir_test,
    dir_class_pred,
    target_names=['down', 'flat', 'up'],
))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000779 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3339
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Direction Accuracy = 0.4209544121567582
[[565 174  60]
 [867 935  79]
 [737 255  79]]
              precision    recall  f1-score   support

        down       0.26      0.71      0.38       799
        flat       0.69      0.50      0.58      1881
          up       0.36      0.07      0.12      1071

    accuracy                           0.42      3751
   macro avg       0.44      0.43      0.36      3751
weighted avg       0.50      0.42      0.41      3751



## Up-Only Direction Model

Train a binary model focused only on whether the next 4-hour move is up.


In [ ]:
y_up = (df_model['future_return_4h'] > direction_threshold).astype(int)
y_up_train = y_up.iloc[:split_index]
y_up_test = y_up.iloc[split_index:]

model_up = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_up.fit(X_train, y_up_train)
up_binary_proba = model_up.predict_proba(X_test)[:, 1]
up_pred = (up_binary_proba >= 0.5).astype(int)

print('Up Accuracy =', accuracy_score(y_up_test, up_pred))
print(confusion_matrix(y_up_test, up_pred))
print(classification_report(
    y_up_test,
    up_pred,
    target_names=['not_up', 'up'],
))

pd.Series(up_binary_proba, name='up_binary_proba').describe()


## Probability-Difference Signals

Use the gap between up probability and down probability instead of taking the largest class directly. This reduces one-sided bias and keeps weak predictions out of the market.

In [ ]:
bt = df_model.iloc[split_index:].copy()
bt['trend_proba'] = proba
bt['up_binary_proba'] = up_binary_proba
bt['down_proba'] = dir_proba[:, signal_to_class[-1]]
bt['flat_proba'] = dir_proba[:, signal_to_class[0]]
bt['up_proba'] = dir_proba[:, signal_to_class[1]]
bt['direction_edge'] = bt['up_proba'] - bt['down_proba']
bt['dir_confidence'] = np.maximum(bt['up_proba'], bt['down_proba'])

def make_probability_edge_signal(
    frame,
    trend_threshold=0.60,
    edge_threshold=0.20,
    min_direction_confidence=0.45,
    allow_long=True,
    allow_short=True,
):
    signal = pd.Series(0, index=frame.index, dtype=int)
    active = (
        (frame['trend_proba'] >= trend_threshold)
        & (frame['dir_confidence'] >= min_direction_confidence)
    )
    if allow_long:
        signal.loc[active & (frame['direction_edge'] >= edge_threshold)] = 1
    if allow_short:
        signal.loc[active & (frame['direction_edge'] <= -edge_threshold)] = -1
    return signal

bt[['trend_proba', 'up_binary_proba', 'down_proba', 'flat_proba', 'up_proba', 'direction_edge', 'dir_confidence']].describe()


## Strategy Comparison

Compare long-only, short-only, and long-short variants across several edge thresholds.

In [67]:
strategy_rows = []
edge_thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]

for edge_threshold in edge_thresholds:
    for name, allow_long, allow_short in [
        ('long_short', True, True),
        ('long_only', True, False),
        ('short_only', False, True),
    ]:
        signal = make_probability_edge_signal(
            bt,
            trend_threshold=0.60,
            edge_threshold=edge_threshold,
            min_direction_confidence=0.45,
            allow_long=allow_long,
            allow_short=allow_short,
        )
        returns = build_long_short_returns(
            bt.assign(trade_signal=signal),
            signal_col='trade_signal',
            return_col='future_return_4h',
            cost_per_trade=0.0002,
        )
        summary = summarize_returns(returns)
        strategy_rows.append({
            'strategy': name,
            'edge_threshold': edge_threshold,
            'long_count': int((signal == 1).sum()),
            'short_count': int((signal == -1).sum()),
            'flat_count': int((signal == 0).sum()),
            **summary,
        })

strategy_results = pd.DataFrame(strategy_rows)
display_cols = [
    'strategy', 'edge_threshold', 'long_count', 'short_count', 'trades',
    'total_return', 'entry_win_rate', 'entry_mean_return',
    'avg_win', 'avg_loss', 'profit_factor', 'max_drawdown',
]
strategy_results.sort_values('total_return', ascending=False)[display_cols]


,strategy,edge_threshold,long_count,short_count,trades,total_return,entry_win_rate,entry_mean_return,avg_win,avg_loss,profit_factor,max_drawdown
1,long_only,0.10,124,0,179.0,0.132087,0.391061,0.000726,0.008072,-0.003991,1.298881,-0.143748
7,long_only,0.20,106,0,158.0,0.113583,0.379747,0.000712,0.007975,-0.003734,1.307497,-0.128240
13,long_only,0.30,76,0,119.0,0.099375,0.369748,0.000821,0.007485,-0.003088,1.421993,-0.090953
4,long_only,0.15,111,0,164.0,0.094456,0.378049,0.000582,0.007867,-0.003847,1.243116,-0.128240
10,long_only,0.25,92,0,140.0,0.066579,0.364286,0.000489,0.007636,-0.003606,1.213376,-0.128240
16,long_only,0.35,58,0,92.0,0.045415,0.336957,0.000506,0.007533,-0.003065,1.248955,-0.072359
15,long_short,0.35,58,1318,1902.0,-0.564250,0.336488,-0.000382,0.009198,-0.005240,0.890132,-0.598916
0,long_short,0.10,124,1456,2164.0,-0.581833,0.341959,-0.000348,0.009266,-0.005343,0.901145,-0.628045
17,short_only,0.35,0,1318,1810.0,-0.583180,0.336464,-0.000427,0.009283,-0.005351,0.879692,-0.614082
3,long_short,0.15,111,1447,2138.0,-0.591985,0.340973,-0.000364,0.009248,-0.005337,0.896525,-0.633938


## Up-Only Strategy Comparison

Compare long-only signals based on the binary up probability, with and without the trend filter.


In [ ]:
def make_up_only_signal(
    frame,
    up_threshold=0.55,
    trend_threshold=None,
):
    signal = pd.Series(0, index=frame.index, dtype=int)
    active = frame['up_binary_proba'] >= up_threshold
    if trend_threshold is not None:
        active = active & (frame['trend_proba'] >= trend_threshold)
    signal.loc[active] = 1
    return signal

up_strategy_rows = []
up_thresholds = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
trend_threshold_options = [None, 0.50, 0.60]

for up_threshold in up_thresholds:
    for trend_threshold in trend_threshold_options:
        signal = make_up_only_signal(
            bt,
            up_threshold=up_threshold,
            trend_threshold=trend_threshold,
        )
        returns = build_long_short_returns(
            bt.assign(trade_signal=signal),
            signal_col='trade_signal',
            return_col='future_return_4h',
            cost_per_trade=0.0002,
        )
        summary = summarize_returns(returns)
        up_strategy_rows.append({
            'strategy': 'up_binary' if trend_threshold is None else f'up_binary_trend_{trend_threshold:.2f}',
            'up_threshold': up_threshold,
            'trend_threshold': trend_threshold,
            'long_count': int((signal == 1).sum()),
            'flat_count': int((signal == 0).sum()),
            **summary,
        })

up_strategy_results = pd.DataFrame(up_strategy_rows)
up_display_cols = [
    'strategy', 'up_threshold', 'trend_threshold', 'long_count', 'trades',
    'total_return', 'entry_win_rate', 'entry_mean_return',
    'avg_win', 'avg_loss', 'profit_factor', 'max_drawdown',
]
up_strategy_results.sort_values('total_return', ascending=False)[up_display_cols]


## Selected Up-Only Backtest

Inspect the best binary up-probability strategy separately from the old multiclass edge strategy.


In [ ]:
best_up = up_strategy_results.sort_values('total_return', ascending=False).iloc[0]
selected_up_signal = make_up_only_signal(
    bt,
    up_threshold=float(best_up['up_threshold']),
    trend_threshold=None if pd.isna(best_up['trend_threshold']) else float(best_up['trend_threshold']),
)
bt['up_only_trade_signal'] = selected_up_signal
selected_up_returns = build_long_short_returns(
    bt,
    signal_col='up_only_trade_signal',
    return_col='future_return_4h',
    cost_per_trade=0.0002,
)
selected_up_equity = equity_curve(selected_up_returns)
selected_up_yearly = yearly_returns(selected_up_returns, bt[df.columns[0]])

print('Selected up-only strategy:')
print(best_up[up_display_cols])
print('\nSignal counts:')
print(bt['up_only_trade_signal'].value_counts().sort_index())
print('\nBacktest summary:')
selected_up_summary = summarize_returns(selected_up_returns)
print(selected_up_summary)
print('\nYearly returns:')
selected_up_yearly


In [ ]:
datetime_col = df.columns[0]
up_only_diagnostics = bt.copy()
up_only_diagnostics['year'] = pd.to_datetime(up_only_diagnostics[datetime_col]).dt.year
up_only_diagnostics['up_only_trade_signal'] = selected_up_signal

up_only_yearly = up_only_diagnostics.groupby('year').agg(
    rows=(datetime_col, 'size'),
    up_signal_count=('up_only_trade_signal', 'sum'),
    actual_up_4h=('future_return_4h', lambda s: (s > direction_threshold).sum()),
    mean_up_binary_proba=('up_binary_proba', 'mean'),
    max_up_binary_proba=('up_binary_proba', 'max'),
    mean_trend_proba=('trend_proba', 'mean'),
)
up_only_yearly['up_signal_rate'] = up_only_yearly['up_signal_count'] / up_only_yearly['rows']
up_only_yearly['actual_up_4h_rate'] = up_only_yearly['actual_up_4h'] / up_only_yearly['rows']
up_only_yearly


## Selected Backtest

Start from the best row in the comparison table, then inspect recent trades. This is still a research result, not a deployable strategy.

In [68]:
best = strategy_results.sort_values('total_return', ascending=False).iloc[0]
selected_signal = make_probability_edge_signal(
    bt,
    trend_threshold=0.60,
    edge_threshold=float(best['edge_threshold']),
    min_direction_confidence=0.45,
    allow_long=best['strategy'] in ['long_short', 'long_only'],
    allow_short=best['strategy'] in ['long_short', 'short_only'],
)
bt['trade_signal'] = selected_signal
selected_returns = build_long_short_returns(
    bt,
    signal_col='trade_signal',
    return_col='future_return_4h',
    cost_per_trade=0.0002,
)
selected_equity = equity_curve(selected_returns)
selected_yearly = yearly_returns(selected_returns, bt[df.columns[0]])

print('Selected strategy:')
print(best[display_cols])
print('\nSignal counts:')
print(bt['trade_signal'].value_counts().sort_index())
print('\nBacktest summary:')
selected_summary = summarize_returns(selected_returns)
print(selected_summary)
print('\nYearly returns:')
selected_yearly


Selected strategy:
strategy             long_only
edge_threshold             0.1
long_count                 124
short_count                  0
trades                   179.0
total_return          0.132087
entry_win_rate        0.391061
entry_mean_return     0.000726
avg_win               0.008072
avg_loss             -0.003991
profit_factor         1.298881
max_drawdown         -0.143748
Name: 1, dtype: object

Signal counts:
trade_signal
0    3627
1     124
Name: count, dtype: int64

Backtest summary:
{'trades': 179.0, 'total_return': 0.13208673472458465, 'win_rate': 0.018661690215942415, 'entry_win_rate': 0.39106145251396646, 'mean_return': 3.466461907697927e-05, 'entry_mean_return': 0.0007264077438980406, 'avg_win': 0.008072464397206167, 'avg_loss': -0.00399124331785947, 'profit_factor': 1.2988813346832921, 'max_drawdown': -0.14374760791397756}

Yearly returns:


,year,return,trades,entry_win_rate
0,2023,0.118739,177,0.389831
1,2024,0.011931,2,0.500000
2,2025,0.000000,0,0.000000
3,2026,0.000000,0,0.000000


## No-Trade Diagnostics

Break down why the selected strategy stops trading in recent years.

In [69]:
datetime_col = df.columns[0]
selected_edge_threshold = float(best['edge_threshold'])

signal_diagnostics = bt.copy()
signal_diagnostics['year'] = pd.to_datetime(signal_diagnostics[datetime_col]).dt.year
signal_diagnostics['pass_trend'] = signal_diagnostics['trend_proba'] >= 0.60
signal_diagnostics['pass_confidence'] = signal_diagnostics['dir_confidence'] >= 0.45
signal_diagnostics['pass_long_edge'] = signal_diagnostics['direction_edge'] >= selected_edge_threshold
signal_diagnostics['pass_short_edge'] = signal_diagnostics['direction_edge'] <= -selected_edge_threshold
signal_diagnostics['long_candidate'] = (
    signal_diagnostics['pass_trend']
    & signal_diagnostics['pass_confidence']
    & signal_diagnostics['pass_long_edge']
)
signal_diagnostics['actual_up_4h'] = signal_diagnostics['future_return_4h'] > direction_threshold
signal_diagnostics['actual_down_4h'] = signal_diagnostics['future_return_4h'] < -direction_threshold

no_trade_yearly = signal_diagnostics.groupby('year').agg(
    rows=(datetime_col, 'size'),
    trend_pass=('pass_trend', 'sum'),
    confidence_pass=('pass_confidence', 'sum'),
    long_edge_pass=('pass_long_edge', 'sum'),
    short_edge_pass=('pass_short_edge', 'sum'),
    long_candidates=('long_candidate', 'sum'),
    actual_up_4h=('actual_up_4h', 'sum'),
    actual_down_4h=('actual_down_4h', 'sum'),
    mean_trend_proba=('trend_proba', 'mean'),
    mean_down_proba=('down_proba', 'mean'),
    mean_up_proba=('up_proba', 'mean'),
    mean_direction_edge=('direction_edge', 'mean'),
)

rate_cols = ['trend_pass', 'confidence_pass', 'long_edge_pass', 'short_edge_pass', 'long_candidates', 'actual_up_4h', 'actual_down_4h']
no_trade_yearly_rates = no_trade_yearly.copy()
for col in rate_cols:
    no_trade_yearly_rates[col + '_rate'] = no_trade_yearly_rates[col] / no_trade_yearly_rates['rows']

no_trade_yearly_rates[[
    'rows', 'trend_pass', 'confidence_pass', 'long_edge_pass', 'short_edge_pass', 'long_candidates',
    'actual_up_4h', 'actual_down_4h', 'mean_trend_proba', 'mean_down_proba', 'mean_up_proba', 'mean_direction_edge',
    'trend_pass_rate', 'long_edge_pass_rate', 'long_candidates_rate', 'actual_up_4h_rate',
]]


,rows,trend_pass,confidence_pass,long_edge_pass,short_edge_pass,long_candidates,actual_up_4h,actual_down_4h,mean_trend_proba,mean_down_proba,mean_up_proba,mean_direction_edge,trend_pass_rate,long_edge_pass_rate,long_candidates_rate,actual_up_4h_rate
year,,,,,,,,,,,,,,,,
2023,666,343,346,221,215,123,177,146,0.523797,0.296598,0.287316,-0.009282,0.515015,0.331832,0.184685,0.265766
2024,1295,542,770,5,1043,1,364,281,0.492282,0.488465,0.090828,-0.397637,0.418533,0.003861,0.000772,0.281081
2025,1289,643,815,6,1102,0,374,264,0.543171,0.534860,0.090516,-0.444344,0.498836,0.004655,0.000000,0.290147
2026,501,269,338,0,461,0,156,108,0.554038,0.582389,0.073630,-0.508758,0.536926,0.000000,0.000000,0.311377


In [70]:
recent_no_trade = signal_diagnostics[
    (signal_diagnostics['year'] >= 2025)
    & (~signal_diagnostics['long_candidate'])
].copy()

recent_no_trade['blocked_by_trend'] = ~recent_no_trade['pass_trend']
recent_no_trade['blocked_by_confidence'] = ~recent_no_trade['pass_confidence']
recent_no_trade['blocked_by_long_edge'] = ~recent_no_trade['pass_long_edge']

blocker_summary = recent_no_trade[[
    'blocked_by_trend', 'blocked_by_confidence', 'blocked_by_long_edge',
]].sum().rename('count').to_frame()
blocker_summary['rate'] = blocker_summary['count'] / len(recent_no_trade)

print(f'Recent no-trade rows: {len(recent_no_trade)}')
display(blocker_summary)

recent_cols = [
    datetime_col, 'close', 'future_return_4h', 'trend_proba', 'down_proba', 'up_proba',
    'direction_edge', 'dir_confidence', 'pass_trend', 'pass_confidence', 'pass_long_edge',
]
recent_no_trade[recent_cols].tail(30)


Recent no-trade rows: 1790


,count,rate
blocked_by_trend,878,0.490503
blocked_by_confidence,637,0.355866
blocked_by_long_edge,1784,0.996648


,日時,close,future_return_4h,trend_proba,down_proba,up_proba,direction_edge,dir_confidence,pass_trend,pass_confidence,pass_long_edge
85672,2026-05-14 09:00:00,29680.3,0.000229,0.727779,0.918980,0.030400,-0.888580,0.918980,True,True,False
85673,2026-05-14 10:00:00,29607.1,0.002591,0.700447,0.914222,0.023610,-0.890611,0.914222,True,True,False
85674,2026-05-14 11:00:00,29586.4,-0.002366,0.584758,0.922479,0.014065,-0.908413,0.922479,False,True,False
85675,2026-05-14 12:00:00,29534.3,0.001534,0.464711,0.864653,0.022658,-0.841995,0.864653,False,True,False
85694,2026-05-15 08:00:00,29687.1,-0.004180,0.199185,0.263019,0.033778,-0.229240,0.263019,False,False,False
85695,2026-05-15 09:00:00,29683.8,-0.019068,0.816860,0.902286,0.021556,-0.880730,0.902286,True,True,False
85696,2026-05-15 10:00:00,29516.4,-0.016374,0.849254,0.788836,0.073715,-0.715120,0.788836,True,True,False
85697,2026-05-15 11:00:00,29579.6,-0.018874,0.863427,0.867692,0.040772,-0.826921,0.867692,True,True,False
85698,2026-05-15 12:00:00,29563.0,-0.018736,0.823590,0.701283,0.105643,-0.595640,0.701283,True,True,False
85717,2026-05-18 08:00:00,29117.8,-0.002160,0.614716,0.633982,0.084501,-0.549482,0.633982,True,True,False


In [71]:
equity_preview = pd.DataFrame({
    df.columns[0]: bt[df.columns[0]].values,
    'strategy_return': selected_returns.values,
    'equity': selected_equity.values,
})
equity_preview.tail(30)


,日時,strategy_return,equity
3721,2026-05-14 09:00:00,0.0,1.132087
3722,2026-05-14 10:00:00,0.0,1.132087
3723,2026-05-14 11:00:00,-0.0,1.132087
3724,2026-05-14 12:00:00,0.0,1.132087
3725,2026-05-15 08:00:00,-0.0,1.132087
3726,2026-05-15 09:00:00,-0.0,1.132087
3727,2026-05-15 10:00:00,-0.0,1.132087
3728,2026-05-15 11:00:00,-0.0,1.132087
3729,2026-05-15 12:00:00,-0.0,1.132087
3730,2026-05-18 08:00:00,-0.0,1.132087


In [72]:
datetime_col = df.columns[0]
bt[[datetime_col, 'close', 'future_return_4h', 'trend_proba', 'down_proba', 'up_proba', 'direction_edge', 'trade_signal']].tail(30)


,日時,close,future_return_4h,trend_proba,down_proba,up_proba,direction_edge,trade_signal
85672,2026-05-14 09:00:00,29680.3,0.000229,0.727779,0.918980,0.030400,-0.888580,0
85673,2026-05-14 10:00:00,29607.1,0.002591,0.700447,0.914222,0.023610,-0.890611,0
85674,2026-05-14 11:00:00,29586.4,-0.002366,0.584758,0.922479,0.014065,-0.908413,0
85675,2026-05-14 12:00:00,29534.3,0.001534,0.464711,0.864653,0.022658,-0.841995,0
85694,2026-05-15 08:00:00,29687.1,-0.004180,0.199185,0.263019,0.033778,-0.229240,0
85695,2026-05-15 09:00:00,29683.8,-0.019068,0.816860,0.902286,0.021556,-0.880730,0
85696,2026-05-15 10:00:00,29516.4,-0.016374,0.849254,0.788836,0.073715,-0.715120,0
85697,2026-05-15 11:00:00,29579.6,-0.018874,0.863427,0.867692,0.040772,-0.826921,0
85698,2026-05-15 12:00:00,29563.0,-0.018736,0.823590,0.701283,0.105643,-0.595640,0
85717,2026-05-18 08:00:00,29117.8,-0.002160,0.614716,0.633982,0.084501,-0.549482,0
